<a href="https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chiemahp/Flyrank-internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns the model and queue outputs into a practical playbook: what to review first, why, and where the recommendations stop being trustworthy.


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The strongest queue items are the ones that combine a high model score with a clear, interpretable reason code: they look like visible pages that are declining and have enough opportunity to justify a refresh review.


In [ ]:
import pandas as pd
from pathlib import Path

root = Path.cwd().resolve()
for parent in [root, *root.parents]:
    if (parent / 'AGENTS.md').exists() and (parent / 'skills').exists():
        root = parent
        break

queue_path = root / 'outputs' / 'refresh_queue_sample.csv'
if not queue_path.exists():
    raise FileNotFoundError('Run the export workflow to create outputs/refresh_queue_sample.csv first')

queue = pd.read_csv(queue_path)
queue = queue.sort_values('final_rank').reset_index(drop=True)

print('queue_rows', len(queue))
print('confidence_counts')
print(queue['confidence'].value_counts().to_dict())
print('\nTop 10 ranked items:')
print(queue[['final_rank', 'content_id', 'confidence', 'suggested_action', 'final_reason_codes']].head(10).to_string(index=False))

# Summarize the most common reasons.
reason_counts = pd.Series(' '.join(queue['final_reason_codes'].fillna('').astype(str)).split('|')).value_counts()
print('\nTop reason codes:')
print(reason_counts.head(10))


queue_rows 200
confidence_counts
{'high': 161, 'medium': 39}

Top 10 ranked items:
 final_rank           content_id confidence       suggested_action                                                                                                                                                   final_reason_codes
          1 content_1f080331fa2b       high refresh_and_review_ctr declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate|engagement_review_candidate
          2 content_6aa43079fb0c       high refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate
          3 content_d6570c51c9bd     medium refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This playbook is intended for a human reviewer who is triaging refresh candidates. It supports prioritization, not automation of content decisions, because the model is trained on observed patterns and the data can only support decision support.


In [ ]:
import pandas as pd
from pathlib import Path

root = Path.cwd().resolve()
for parent in [root, *root.parents]:
    if (parent / 'AGENTS.md').exists() and (parent / 'skills').exists():
        root = parent
        break

queue_path = root / 'outputs' / 'refresh_queue_sample.csv'
queue = pd.read_csv(queue_path)

# A simple usage note grounded in the actual queue columns.
reviewable = queue[queue['confidence'].isin(['high', 'medium'])].copy()
print('reviewable_rows', len(reviewable))
print('high_confidence_rows', int((queue['confidence'] == 'high').sum()))
print('average_model_probability', round(reviewable['best_model_probability'].mean(), 3))
print('average_score', round(reviewable['final_refresh_score'].mean(), 1))


reviewable_rows 200
high_confidence_rows 161
average_model_probability 0.802
average_score 77.9


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

A reviewer should verify the page still has a real business need, that the reason code matches the current page state, and that the page has not already been updated recently. The model should not be used to auto-publish, auto-delete, or auto-rewrite content without human review.


In [ ]:
import pandas as pd
from pathlib import Path

root = Path.cwd().resolve()
for parent in [root, *root.parents]:
    if (parent / 'AGENTS.md').exists() and (parent / 'skills').exists():
        root = parent
        break

queue_path = root / 'outputs' / 'refresh_queue_sample.csv'
queue = pd.read_csv(queue_path)

# Show the most common reasons that should trigger manual review.
manual_review_terms = ['ctr_review_candidate', 'engagement_review_candidate', 'model_decline_risk']
for term in manual_review_terms:
    count = queue['final_reason_codes'].astype(str).str.contains(term).sum()
    print(term, count)

print('\nExample no-go situations:')
print('- Auto-publish without a human check on page intent and freshness.')
print('- Auto-rewrite content that is still receiving traffic or has recent updates.')
print('- Treat model output as a guarantee of decline or business value.')


ctr_review_candidate 130
engagement_review_candidate 78
model_decline_risk 200

Example no-go situations:
- Auto-publish without a human check on page intent and freshness.
- Auto-rewrite content that is still receiving traffic or has recent updates.
- Treat model output as a guarantee of decline or business value.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The queue should be monitored if the mix of actions shifts, the model scores stop aligning with observed decline, or the top reasons become dominated by stale or low-traffic cases. A retraining trigger is any sustained drop in ranking quality or a shift in the target population.


In [ ]:
import pandas as pd
from pathlib import Path

root = Path.cwd().resolve()
for parent in [root, *root.parents]:
    if (parent / 'AGENTS.md').exists() and (parent / 'skills').exists():
        root = parent
        break

queue_path = root / 'outputs' / 'refresh_queue_sample.csv'
queue = pd.read_csv(queue_path)

# Simple operational checks from the queue itself.
summary = {
    'rows': len(queue),
    'high_confidence': int((queue['confidence'] == 'high').sum()),
    'avg_score': round(queue['final_refresh_score'].mean(), 2),
    'avg_model_probability': round(queue['best_model_probability'].mean(), 3),
}
print(summary)

print('\nIf any of these happen, review the model:')
print('- high-confidence queue share changes sharply')
print('- model probability no longer separates declining from stable pages')
print('- top reason codes shift away from visible decline and toward stale or unrelated cases')


{'rows': 200, 'high_confidence': 161, 'avg_score': np.float64(77.95), 'avg_model_probability': np.float64(0.802)}

If any of these happen, review the model:
- high-confidence queue share changes sharply
- model probability no longer separates declining from stable pages
- top reason codes shift away from visible decline and toward stale or unrelated cases


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The notebook writes a compact queue export that can be reused by the paper and report pipeline.


In [ ]:
import pandas as pd
from pathlib import Path

root = Path.cwd().resolve()
for parent in [root, *root.parents]:
    if (parent / 'AGENTS.md').exists() and (parent / 'skills').exists():
        root = parent
        break

queue_path = root / 'outputs' / 'refresh_queue_sample.csv'
out_dir = root / 'work' / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

queue = pd.read_csv(queue_path)
export_path = out_dir / 'action_playbook_queue.csv'
queue[['final_rank', 'content_id', 'confidence', 'suggested_action', 'final_reason_codes', 'best_model_probability', 'final_refresh_score']].head(50).to_csv(export_path, index=False)
print(f'Wrote {export_path}')
print(queue[['final_rank', 'content_id', 'confidence', 'suggested_action', 'final_reason_codes']].head(10).to_string(index=False))


Wrote D:\Flyrank-internship-ml\work\outputs\action_playbook_queue.csv
 final_rank           content_id confidence       suggested_action                                                                                                                                                   final_reason_codes
          1 content_1f080331fa2b       high refresh_and_review_ctr declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate|engagement_review_candidate
          2 content_6aa43079fb0c       high refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate
          3 content_d6570c51c9bd     medium refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate
  

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.